In [1]:
import nvdlib
import json
import pandas as pd
import numpy as np
import re
import ast
from tqdm.notebook import tqdm
import sys
import string
import nltk
from nltk.tokenize import word_tokenize
import os
import fnmatch
import warnings
from pprint import pprint
warnings.filterwarnings('ignore')

In [107]:
from nltk.corpus import stopwords
tqdm.pandas()
directory_path = "/home/umd-user/Desktop/navex_project/navex_tests/osCommerce"
appName = 'oscommerce'
extension = ''
appVersion = '2.3.4.1'

nltk.download('stopwords')
nltk.download('punkt')

stopwords = list(filter(lambda x: len(x)>1, stopwords.words('english')))
punctuation = set(string.punctuation)
punctuation.remove('(')
punctuation.remove(')')
punctuation.remove('$')

[nltk_data] Downloading package stopwords to /home/umd-
[nltk_data]     user/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/umd-user/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [108]:
def getFiles(description):
    # extensions = ['php', 'html', 'js']
    extensions = ['php']
    result = []
    for ext in extensions:
        result += re.findall(r'\b\w+\.' + ext + r'\b', description, re.IGNORECASE)
    return result

def getVersions(description):
    result = re.findall(r'\d+\.\d+\.\d+', description)
    for version in result: description = description.replace(version, '')
    result += re.findall(r'\d+\.\d+', description)
    if result==[]:
        return ['0']
    else: return result

def getVulnerability(description):
    desc = description.lower()
    if "sql" in desc:
        return "SQL Injection"
    elif "xss" in desc or "cross-site scripting" in desc or "cross site scripting" in desc:
        return "XSS"
    elif "file upload" in desc or "file inclusion" in desc:
        return "File Inclusion"
    elif "file access" in desc:
        return "File Access"
    elif "session" in desc:
        return "Session Fixation"
    elif "code injection" in desc:
        return "Code Injection"
    elif "command" in desc:
        return "Command Execution"
    # elif "csrf" in desc or "request forgery" in desc:
    #     return "CSRF"
    else:
        return "NA"
    
# Compute whether appVersion was released before cveVersion (inclusive)
def compareVersions(appVersion, cveVersions):
    flag = False
    if (len(cveVersions)==0 or len(appVersion)==0 or cveVersions==['0']): flag = True
    for cveVersion in cveVersions:
        v1 = list(map(int, appVersion.split('.')))
        v2 = list(map(int, cveVersion.split('.')))
        size = min(len(v1), len(v2))
        v1 = v1[:size]
        v2 = v2[:size]
        if (v1 <= v2): flag = True
    return flag

def getParameters(description):
    words = word_tokenize(description)
    words = [word for word in words if word.lower() not in stopwords and word not in punctuation]
    indices = [i for i in range(1, len(words)) if words[i] == "parameter" or words[i] == "parameters"]
    params = [words[i-1] for i in indices]
    indices = [i for i in range(1, len(words)-2) if (words[i-1]=='(' and words[i].isdigit() and words[i+1]==')')]
    params += [words[i+2] for i in indices]
    indices = [i for i in range(len(words)-1) if words[i] == "function" or words[i]=="$"]
    params += [words[i+1] for i in indices]
    params = [param.replace('"', '') for param in params if ('.php' not in param and '/' not in param)]
    return list(set(params))

def getCVEFromNavex(cve, file):
    flag = False
    for index, row in cve.iterrows():
        for fileName in row['filenames']:
            if fileName in file:
                flag = True
                cve_id = row['id']
                break
    if not flag: cve_id = 'NA'
    return cve_id

def filterOnFiles(fileNames):
    if fileNames == []: return True
    for root, dirs, files in os.walk(directory_path):
        for fileName in fileNames:
            for filename in fnmatch.filter(files, fileName):
                # file_path = os.path.join(root, filename)
                return True
    return False

def getFilesFromParameters(parameters):
    file_paths = []
    pattern = r'(?:' + '|'.join(re.escape(parameter) for parameter in parameters) + r')\b'
    if parameters == []: return file_paths
    for root, dirs, files in os.walk(directory_path):
        for filename in files:
            file_path = os.path.join(root, filename)
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
                content = file.read()
                if re.search(pattern, content):
                    file_paths.append(file_path)
    return list(set(file_paths))

In [109]:
r = nvdlib.searchCVE(keywordSearch=appName)
jsonFormattedCVE = json.dumps(ast.literal_eval(str(r)))

In [110]:
cve = pd.read_json(jsonFormattedCVE)

cve['descriptions'] = cve['descriptions'].apply(lambda descriptions: list(filter(lambda x: x["lang"]=="en", descriptions)))
cve['descriptions'] = cve['descriptions'].apply(lambda descriptions: descriptions[0]['value'])
cve['versions'] = cve['descriptions'].apply(getVersions)
cve['filenames'] = cve['descriptions'].apply(getFiles)
cve['cve_vulnerability'] = cve['descriptions'].apply(getVulnerability)
cve['parameters'] = cve['descriptions'].apply(getParameters)
cve['relevant_version'] = cve['versions'].apply(lambda x: compareVersions(appVersion, x))# and compareVersions(x[0], ['2.1.4']))

cve = cve[cve['cve_vulnerability']!='NA']
cve = cve[cve['relevant_version']==True]

cve = cve[cve['filenames'].apply(filterOnFiles)]
cve = cve[cve['parameters'].apply(lambda params: getFilesFromParameters(params) != [])]
cve = cve[cve['filenames'].map(lambda x: x!=[]) | cve['parameters'].map(lambda x: x!=[])]
# cve['filenames'] = cve.apply(lambda x: x.filenames if x.filenames!=[] else list(map(lambda x: x.split('/')[-1], getFilesFromParameters(x.parameters))), axis=1)

cve = cve.loc[:, ['id', 'cve_vulnerability', 'versions', 'filenames', 'parameters', 'descriptions']]

cve.to_excel(f"/home/umd-user/Desktop/navex_project/navex_utils/cve/{appName}-{extension}cve.xlsx")
cve.shape

(3, 6)

In [111]:
cve

,id,cve_vulnerability,versions,filenames,parameters,descriptions
15,CVE-2006-6534,XSS,[3.0],"[modules.php, customers.php, languages_definit...","[lID, pID, selected_box, set]",Multiple cross-site scripting (XSS) vulnerabil...
28,CVE-2012-1792,XSS,[3.0.2],"[DBCheck.php, index.php, index.php]",[name],Cross-site scripting (XSS) vulnerability in os...
49,CVE-2022-35212,XSS,[2.3.4],[],[tep_db_error],osCommerce2 before v2.3.4.1 was discovered to ...


In [112]:
# Output of the navex joern extension
navex = pd.read_json(f'/home/umd-user/Desktop/navex_project/navex_utils/paths/{appName}-{extension}output.json')
navex.shape

(52826, 8)

In [113]:
def getCVEidsFromPath(row, pathRow):
    flag = False
    cve_id = np.nan
    if type(row['filenames'])!=list: files = ast.literal_eval(row['filenames'])
    else: files = row['filenames']
    if type(row['parameters'])!=list: cveParams = ast.literal_eval(row['parameters'])
    else: cveParams = row['parameters']
    if not files: files = ['']
    
    for fileName in files:
        if row['cve_vulnerability'] == pathRow['vulnerability']: 
            if not cveParams and fileName in pathRow.loc['filename']:
                flag = True
            else:
                for param in cveParams:
                    if (fileName in pathRow.loc['filename']) or ((('$' + param.lower().replace('$','')) in pathRow['code'].lower()) or (param == pathRow['methodname'])):
                        flag = True
            if flag:
                    cve_id = row['id']
    return cve_id

In [114]:
CVE_ids = navex.progress_apply(lambda navexRow: list(cve.apply(lambda x: getCVEidsFromPath(x, navexRow), axis=1).dropna().unique()), axis=1)
navex['CVE_ids'] = CVE_ids
navex.head()

  0%|          | 0/52826 [00:00<?, ?it/s]

,pathid,vulnerability,nodeid,methodname,filename,linenumber,code,sanitized,CVE_ids
0,1,XSS,1241,<global>,osCommerce/catalog/account_history.php,60,"$HTTP_GET_VARS[""page""]",FALSE,[CVE-2022-35212]
1,1,XSS,1247,<global>,osCommerce/catalog/account_history.php,60,"$HTTP_GET_VARS[""page""]",FALSE,[CVE-2022-35212]
2,1,XSS,1245,<global>,osCommerce/catalog/account_history.php,60,"""page="" . $HTTP_GET_VARS[""page""]",FALSE,[CVE-2022-35212]
3,1,XSS,1239,<global>,osCommerce/catalog/account_history.php,60,"isset($HTTP_GET_VARS[""page""]) ? ""page="" . $HTT...",FALSE,[CVE-2022-35212]
4,1,XSS,1238,<global>,osCommerce/catalog/account_history.php,60,"isset($HTTP_GET_VARS[""page""]) ? ""page="" . $HTT...",FALSE,[CVE-2022-35212]


In [115]:
# Check rows that matched with a CVE
emptyList = pd.Series([np.nan] * len(navex['CVE_ids'])).fillna('[]')
navex[navex['CVE_ids'].astype(str) != emptyList].explode('CVE_ids').head()

,pathid,vulnerability,nodeid,methodname,filename,linenumber,code,sanitized,CVE_ids
0,1,XSS,1241,<global>,osCommerce/catalog/account_history.php,60,"$HTTP_GET_VARS[""page""]",FALSE,CVE-2022-35212
1,1,XSS,1247,<global>,osCommerce/catalog/account_history.php,60,"$HTTP_GET_VARS[""page""]",FALSE,CVE-2022-35212
2,1,XSS,1245,<global>,osCommerce/catalog/account_history.php,60,"""page="" . $HTTP_GET_VARS[""page""]",FALSE,CVE-2022-35212
3,1,XSS,1239,<global>,osCommerce/catalog/account_history.php,60,"isset($HTTP_GET_VARS[""page""]) ? ""page="" . $HTT...",FALSE,CVE-2022-35212
4,1,XSS,1238,<global>,osCommerce/catalog/account_history.php,60,"isset($HTTP_GET_VARS[""page""]) ? ""page="" . $HTT...",FALSE,CVE-2022-35212


In [117]:
navex = navex.explode('CVE_ids')
navex.shape

(54461, 9)

In [118]:
navex = navex.merge(cve, how='left', left_on='CVE_ids', right_on='id').drop(columns=['id', 'cve_vulnerability'])

In [119]:
matchedCVEs = {e for e in navex['CVE_ids'].dropna()}
print("Number of exploit matches:", len(matchedCVEs), "out of #" + str(len(cve['id'])), "CVEs")
pprint(matchedCVEs)

Number of exploit matches: 3 out of #3 CVEs
{'CVE-2022-35212', 'CVE-2012-1792', 'CVE-2006-6534'}


In [120]:
# Save data to excel
dfToSave = navex.dropna()
dfToSave.reset_index(inplace=True)
dfToSave.to_excel(f"/home/umd-user/Desktop/navex_project/navex_utils/cve/{appName}-{extension}navex.xlsx")
dfToSave.shape

(45772, 14)

In [121]:
# xssPaths = navex[navex['vulnerability'] == 'XSS']
xssPaths = navex
acrossDb = xssPaths[xssPaths['code'].apply(lambda x: '_query' in x)]
notDb = xssPaths[xssPaths['pathid'].apply(lambda x: x not in acrossDb['pathid'])]
acrossDb['CVE_ids'].unique().shape, notDb['CVE_ids'].unique().shape, xssPaths['CVE_ids'].unique().shape

((3,), (4,), (4,))